In [0]:
# =====================================================
# Component 5: Complete Orchestration Example
# =====================================================
# End-to-end ingestion orchestrator with error handling

import traceback
from datetime import datetime

def ingest_object(object_row, job_run_id, connection_params=None):
    """
    Execute full ingestion workflow for a single object
    
    Args:
        object_row: Row from ObjectsList
        job_run_id: Databricks Job Run ID or unique execution identifier
        connection_params: Dict of connection configuration (optional, can use secrets)
    
    Returns:
        dict: Execution summary with status and metrics
    """
    object_id = object_row.ObjectID
    object_name = object_row.ObjectName
    source_system = object_row.SourceSystem
    target_path = object_row.StoragePath
    target_table = f"{object_row.TargetSchema}.{object_row.TargetTableName}"
    load_incremental = object_row.LoadIncremental
    
    print(f"\n{'='*80}")
    print(f"Starting ingestion: {source_system}.{object_name} → {target_table}")
    print(f"{'='*80}")
    
    extract_id = None
    
    try:
        # 1. Start audit logging
        extract_id = start_extract_audit(object_id, job_run_id, load_incremental)
        print(f"[1/6] ✓ Audit record created (ExtractID: {extract_id})")
        
        # 2. Load column metadata
        fields_df = load_object_fields(object_id)
        print(f"[2/6] ✓ Loaded {fields_df.count()} column definitions")
        
        # 3. Build dynamic query
        source_query = build_source_query(object_row, fields_df)
        print(f"[3/6] ✓ Generated source query")
        print(f"      Load Type: {'INCREMENTAL' if load_incremental else 'FULL'}")
        if load_incremental and object_row.LastWatermarkValue:
            print(f"      Last Watermark: {object_row.LastWatermarkValue}")
        
        # 4. Extract data from source
        print(f"[4/6] ⌛ Extracting from {source_system}...")
        start_time = time.time()
        extracted_df = SourceConnector.extract(source_system, source_query, connection_params)
        
        # Add metadata columns
        extracted_df = extracted_df \
            .withColumn("_extract_timestamp", current_timestamp()) \
            .withColumn("_source_system", expr(f"'{source_system}'")) \
            .withColumn("_extract_id", expr(f"{extract_id}"))
        
        rows_extracted = extracted_df.count()
        extraction_duration = time.time() - start_time
        print(f"      ✓ Extracted {rows_extracted:,} rows in {extraction_duration:.2f}s")
        
        if rows_extracted == 0:
            print(f"      ⚠ No new data to process")
            complete_extract_audit(extract_id, 0, 0, target_path, "N/A", status='SKIPPED', error_msg="No new data")
            return {"status": "SKIPPED", "rows": 0}
        
        # 5. Write to target (Delta Lake)
        print(f"[5/6] ⌛ Writing to {target_table}...")
        write_start = time.time()
        
        # Ensure target schema exists
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {object_row.TargetSchema}")
        
        if load_incremental and object_row.PrimaryKeys:
            # MERGE for incremental with deduplication
            primary_keys = object_row.PrimaryKeys
            merge_condition = " AND ".join([f"target.{pk} = source.{pk}" for pk in primary_keys])
            
            # Write to temp table first
            temp_table = f"{target_table}_temp"
            extracted_df.write.mode("overwrite").saveAsTable(temp_table)
            
            spark.sql(f"""
                MERGE INTO {target_table} AS target
                USING {temp_table} AS source
                ON {merge_condition}
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            
            spark.sql(f"DROP TABLE IF EXISTS {temp_table}")
            rows_copied = rows_extracted  # Approximation
        else:
            # Full load - overwrite
            extracted_df.write \
                .format("delta") \
                .mode("overwrite") \
                .option("path", target_path) \
                .saveAsTable(target_table)
            rows_copied = rows_extracted
        
        write_duration = time.time() - write_start
        print(f"      ✓ Written {rows_copied:,} rows in {write_duration:.2f}s")
        
        # 6. Update watermark for incremental loads
        if load_incremental and object_row.WatermarkColumn:
            new_watermark = calculate_watermark(extracted_df, object_row.WatermarkColumn)
            if new_watermark:
                update_watermark(object_id, new_watermark)
                print(f"[6/6] ✓ Watermark updated: {new_watermark}")
        else:
            print(f"[6/6] ✓ Full load complete (no watermark update)")
        
        # Complete audit record
        complete_extract_audit(
            extract_id, 
            rows_extracted, 
            rows_copied, 
            target_path, 
            f"delta_v{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            status='SUCCESS'
        )
        
        print(f"\n✓✓✓ SUCCESS: {object_name} ingested successfully \n")
        
        return {
            "status": "SUCCESS",
            "object_id": object_id,
            "rows_extracted": rows_extracted,
            "rows_copied": rows_copied,
            "extract_id": extract_id
        }
        
    except Exception as e:
        error_msg = f"{str(e)}\n{traceback.format_exc()}"
        print(f"\n✗ ERROR: {str(e)}\n")
        
        if extract_id:
            complete_extract_audit(
                extract_id, 
                0, 
                0, 
                target_path, 
                "N/A",
                status='FAILED',
                error_msg=error_msg[:2000]  # Limit error message length
            )
        
        return {
            "status": "FAILED",
            "object_id": object_id,
            "error": str(e),
            "extract_id": extract_id
        }

def ingest_load_group(load_group, job_run_id=None, connection_params=None):
    """
    Orchestrate ingestion for all objects in a LoadGroup
    
    Args:
        load_group (int): LoadGroup number to process
        job_run_id (str): Optional Job Run ID (auto-generated if not provided)
        connection_params (dict): Connection parameters for source systems
    
    Returns:
        list: Execution summary for all objects
    """
    if not job_run_id:
        job_run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    print(f"\n{'#'*80}")
    print(f"# LOAD GROUP {load_group} EXECUTION - Job Run: {job_run_id}")
    print(f"{'#'*80}\n")
    
    # Load objects in this load group
    objects = load_active_objects(load_group=load_group).collect()
    
    results = []
    for obj in objects:
        result = ingest_object(obj, job_run_id, connection_params)
        results.append(result)
    
    # Summary
    success_count = sum(1 for r in results if r["status"] == "SUCCESS")
    failed_count = sum(1 for r in results if r["status"] == "FAILED")
    skipped_count = sum(1 for r in results if r["status"] == "SKIPPED")
    total_rows = sum(r.get("rows_copied", 0) for r in results)
    
    print(f"\n{'#'*80}")
    print(f"# LOAD GROUP {load_group} SUMMARY")
    print(f"{'#'*80}")
    print(f"Total Objects: {len(results)}")
    print(f"Success: {success_count} | Failed: {failed_count} | Skipped: {skipped_count}")
    print(f"Total Rows Loaded: {total_rows:,}")
    print(f"{'#'*80}\n")
    
    return results

print("Orchestration functions loaded. Ready to execute ingestion workflows.")

---
## Production-Grade Considerations

### 1. **Composite Primary Keys**
- **Implementation**: The `PrimaryKeys` column stores `ARRAY<STRING>` to support both single and composite keys
- **MERGE Logic**: Build dynamic merge conditions: `key1 = key1 AND key2 = key2`
- **Example**:
  ```sql
  PrimaryKeys = array('MATNR', 'WERKS') -- SAP Material + Plant
  ```

### 2. **Watermark State Management**
- **Single Column**: `LastWatermarkValue = '2026-09-01 15:30:00'`
- **Composite Column**: Concatenate with delimiter: `'2026-09-01_12345'` (timestamp + ID)
- **Reset Strategy**: Set `LastWatermarkValue = NULL` to trigger full reload
- **Idempotency**: Re-running same watermark window produces same results (no duplicates)

### 3. **Schema Drift Handling**
- **Option A - Schema Evolution (Permissive)**:
  ```python
  .option("mergeSchema", "true")
  ```
  Automatically adds new columns from source
  
- **Option B - Schema Enforcement (Strict)**:
  ```python
  spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")
  ```
  Fail fast on schema mismatch, update `ObjectFields` first
  
- **Recommendation**: Use **strict mode** with automated schema change detection:
  1. Compare source schema vs. `ObjectFields` metadata
  2. Log discrepancies to audit table
  3. Require explicit approval to update `ObjectFields`
  4. Re-run ingestion after metadata sync

### 4. **Incremental Load Patterns**
- **Append-Only**: Use `LAST_UPDATE_DATE > watermark`
- **Update Detection**: Use `LAST_MODIFIED_TIMESTAMP > watermark` with MERGE
- **Hard Deletes**: Join source vs. target, identify missing keys, apply DELETE
- **Soft Deletes**: Check `IS_DELETED` flag from source, set `_is_deleted = 1` in target

### 5. **Parallel Execution**
- **LoadGroup Strategy**:
  - Group 1: High-priority transactional data (real-time SLAs)
  - Group 2: Master data (daily batch)
  - Group 3: Historical archives (weekly)
- **Databricks Jobs**: Create separate tasks per LoadGroup with dependencies
- **Concurrency**: Use Spark `threadPoolExecutor` for within-group parallelism

### 6. **Error Recovery**
- **Transient Failures**: Implement exponential backoff with `max_retries = 3`
- **Partial Failures**: LoadGroup continues even if one object fails
- **Alerting**: Send notifications on `ExtractionStatus = 'FAILED'`
- **Replay**: Query `ObjectsExtract` by `JobRunID` to identify failures, re-run specific objects

### 7. **Data Quality Gates**
- Add validation step before writing:
  ```python
  # Row count sanity check
  if load_incremental and rows_extracted > (historical_avg * 3):
      raise ValueError("Anomalous row count detected")
  
  # Null check on critical columns
  null_count = df.filter(col("customer_id").isNull()).count()
  if null_count > 0:
      raise ValueError(f"{null_count} null PKs detected")
  ```

### 8. **Secrets Management**
- **Never hardcode credentials** in notebooks or metadata tables
- Use **Databricks Secrets** for all connection parameters:
  ```python
  dbutils.secrets.get(scope="oracle_prod", key="jdbc_url")
  ```
- Store scope names in `ObjectsList.SourceSystem` mapping table

### 9. **Unity Catalog Integration**
- **Lineage Tracking**: Delta tables in UC automatically track column-level lineage
- **Data Classification**: Apply tags to `ObjectsList.TargetTableName` for PII columns
- **Access Control**: Grant READ on bronze layer, WRITE restricted to service principals

### 10. **Monitoring & Observability**
- **Key Metrics Dashboard**:
  - Ingestion success rate by source system
  - Average extraction duration trends
  - Row count anomaly detection
  - Failed object drill-down
- **Query Example**:
  ```sql
  SELECT 
    DATE(StartTime) as load_date,
    SourceSystem,
    COUNT(*) as total_runs,
    SUM(CASE WHEN ExtractionStatus = 'SUCCESS' THEN 1 ELSE 0 END) as success_count,
    AVG(Duration) as avg_duration_sec,
    SUM(RowsCopied) as total_rows
  FROM mde_dev.MetadataExplorer.ObjectsExtract oe
  JOIN mde_dev.MetadataExplorer.ObjectsList ol ON oe.ObjectID = ol.ObjectID
  WHERE StartTime >= CURRENT_DATE - INTERVAL 7 DAYS
  GROUP BY 1, 2
  ORDER BY 1 DESC, 2
  ```

### 11. **Advanced Patterns**
- **Partitioning**: Add `PartitionColumn` to `ObjectsList`, apply `.partitionBy()` during write
- **Z-Ordering**: Add `ZOrderColumns ARRAY<STRING>`, run `OPTIMIZE ... ZORDER BY` post-load
- **Liquid Clustering**: For high-cardinality keys, use `CLUSTER BY` instead of partitioning
- **Change Data Feed**: Enable on target tables for downstream CDC consumers:
  ```sql
  ALTER TABLE bronze.oracle_ar_customers SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
  ```

---
## Example Execution

### Running the Framework

```python
# Execute LoadGroup 1 (high-priority objects)
results = ingest_load_group(
    load_group=1, 
    job_run_id="JOB_12345_RUN_67890",
    connection_params={
        "oracle": {
            "jdbc_url": dbutils.secrets.get("oracle_prod", "jdbc_url"),
            "username": dbutils.secrets.get("oracle_prod", "username"),
            "password": dbutils.secrets.get("oracle_prod", "password")
        }
    }
)
```

### Databricks Workflow Integration

**Job Configuration**:
- **Task 1**: Ingest LoadGroup 1 (transactional data) - Notebook: `ingest_load_group(1)`
- **Task 2**: Ingest LoadGroup 2 (master data) - Notebook: `ingest_load_group(2)` [depends on Task 1]
- **Task 3**: Data Quality Checks - SQL: Query `ObjectsExtract` for failures
- **Task 4**: Send Alerts - Notebook: Email/Slack notification on failures

### Manual Object Execution

```python
# Ingest a single object by ObjectID
single_object = spark.sql(
    "SELECT * FROM mde_dev.MetadataExplorer.ObjectsList WHERE ObjectID = 1"
).first()

result = ingest_object(
    single_object, 
    job_run_id="MANUAL_TEST_001"
)
```

### Querying Audit History

```sql
-- View recent extraction history
SELECT 
    ol.ObjectName,
    ol.SourceSystem,
    oe.StartTime,
    oe.Duration,
    oe.RowsCopied,
    oe.ExtractionStatus,
    oe.ErrorMessage
FROM mde_dev.MetadataExplorer.ObjectsExtract oe
JOIN mde_dev.MetadataExplorer.ObjectsList ol ON oe.ObjectID = ol.ObjectID
WHERE oe.StartTime >= CURRENT_TIMESTAMP() - INTERVAL 24 HOURS
ORDER BY oe.StartTime DESC;
```

---
## PySpark Framework Architecture

### Design Principles:
1. **Metadata-Driven**: All ingestion logic reads from `ObjectsList` and `ObjectFields`
2. **Dynamic Query Generation**: Column selection and type casting driven by metadata
3. **Watermark Management**: State tracking for incremental loads
4. **Audit by Default**: Every extraction logged to `ObjectsExtract`
5. **Idempotent**: Re-runnable with proper deduplication

### Pattern Overview:
```
Metadata Loader → Dynamic Query Builder → Source Connector → Transform & Write → Audit Logger
```

In [0]:
# =====================================================
# Component 1: Metadata Loader
# =====================================================
# Loads active objects and their column definitions from control tables

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, expr, concat_ws
from datetime import datetime
import json

def load_active_objects(load_group=None):
    """
    Load active objects from ObjectsList filtered by LoadGroup (optional)
    
    Args:
        load_group (int): Optional LoadGroup filter for parallel execution
    
    Returns:
        DataFrame: Active objects with their configuration
    """
    query = """
        SELECT 
            ObjectID,
            ObjectSchema,
            ObjectName,
            SourceSystem,
            LoadIncremental,
            LoadGroup,
            LoadOrder,
            TargetSchema,
            TargetTableName,
            PrimaryKeys,
            WatermarkColumn,
            LastWatermarkValue,
            HardDelete,
            SoftDelete,
            StoragePath
        FROM mde_dev.MetadataExplorer.ObjectsList
        WHERE IncludeInLoad = 1
    """
    
    if load_group:
        query += f" AND LoadGroup = {load_group}"
    
    query += " ORDER BY LoadGroup, LoadOrder"
    
    return spark.sql(query)

def load_object_fields(object_id):
    """
    Load column metadata for a specific object
    
    Args:
        object_id (int): ObjectID from ObjectsList
    
    Returns:
        DataFrame: Column definitions with data types and flags
    """
    return spark.sql(f"""
        SELECT 
            ColumnName,
            TargetColumnName,
            DataType,
            PrimaryKey,
            OrdinalPosition
        FROM mde_dev.MetadataExplorer.ObjectFields
        WHERE ObjectID = {object_id}
          AND IncludeInLoad = 1
        ORDER BY OrdinalPosition
    """)

# Example usage
objects_df = load_active_objects(load_group=1)
print(f"Loaded {objects_df.count()} active objects for extraction")
objects_df.display()

In [0]:
# =====================================================
# Component 2: Dynamic Query Builder
# =====================================================
# Generates source-specific SELECT statements with column projection and type casting

def build_select_clause(fields_df):
    """
    Build dynamic SELECT clause with type casting from metadata
    
    Args:
        fields_df: DataFrame from load_object_fields()
    
    Returns:
        str: SQL SELECT clause with CAST operations
    """
    fields = fields_df.collect()
    
    select_items = []
    for field in fields:
        source_col = field.ColumnName
        target_col = field.TargetColumnName
        data_type = field.DataType
        
        # Handle type casting
        if data_type.upper() == 'TIMESTAMP':
            cast_expr = f"CAST({source_col} AS TIMESTAMP) AS {target_col}"
        elif data_type.upper().startswith('DECIMAL'):
            cast_expr = f"CAST({source_col} AS {data_type}) AS {target_col}"
        elif data_type.upper() in ('INTEGER', 'BIGINT', 'INT', 'LONG'):
            cast_expr = f"CAST({source_col} AS BIGINT) AS {target_col}"
        elif data_type.upper() in ('DOUBLE', 'FLOAT'):
            cast_expr = f"CAST({source_col} AS DOUBLE) AS {target_col}"
        elif data_type.upper() == 'DATE':
            cast_expr = f"CAST({source_col} AS DATE) AS {target_col}"
        elif data_type.upper() == 'BOOLEAN':
            cast_expr = f"CAST({source_col} AS BOOLEAN) AS {target_col}"
        else:  # Default to STRING
            cast_expr = f"CAST({source_col} AS STRING) AS {target_col}"
        
        select_items.append(cast_expr)
    
    return ",\n    ".join(select_items)

def build_source_query(object_row, fields_df):
    """
    Build complete source extraction query with optional incremental filter
    
    Args:
        object_row: Row from ObjectsList
        fields_df: DataFrame from load_object_fields()
    
    Returns:
        str: Complete SQL query for source extraction
    """
    select_clause = build_select_clause(fields_df)
    
    # Build base query
    full_object_name = f"{object_row.ObjectSchema}.{object_row.ObjectName}" if object_row.ObjectSchema else object_row.ObjectName
    
    query = f"""
    SELECT 
        {select_clause}
    FROM {full_object_name}
    """
    
    # Add incremental filter if configured
    if object_row.LoadIncremental == 1 and object_row.WatermarkColumn:
        watermark_cols = object_row.WatermarkColumn
        last_value = object_row.LastWatermarkValue
        
        if last_value:
            # Handle single or composite watermark columns
            if len(watermark_cols) == 1:
                query += f"\n    WHERE {watermark_cols[0]} > '{last_value}'"
            else:
                # Composite watermark (concatenated comparison)
                watermark_concat = " || '_' || ".join(watermark_cols)
                query += f"\n    WHERE {watermark_concat} > '{last_value}'"
    
    return query

# Example: Generate query for first active object
sample_object = objects_df.first()
if sample_object:
    sample_fields = load_object_fields(sample_object.ObjectID)
    generated_query = build_source_query(sample_object, sample_fields)
    
    print("Generated Source Query:")
    print("="*60)
    print(generated_query)
    print("="*60)

In [0]:
# =====================================================
# Component 3: Source Connector & Data Extraction
# =====================================================
# Abstracts connection logic for different source systems

from pyspark.sql import DataFrame
import time

class SourceConnector:
    """
    Factory pattern for connecting to heterogeneous sources
    Extend this class to add new source system types
    """
    
    @staticmethod
    def extract(source_system, query, connection_params=None):
        """
        Execute extraction based on source system type
        
        Args:
            source_system (str): 'Oracle', 'SAP', 'Salesforce', etc.
            query (str): SQL query or API filter
            connection_params (dict): Connection configuration
        
        Returns:
            DataFrame: Extracted data
        """
        source_system = source_system.upper()
        
        if source_system == 'ORACLE':
            return SourceConnector._extract_jdbc(query, 'oracle', connection_params)
        
        elif source_system == 'SAP':
            return SourceConnector._extract_jdbc(query, 'sap', connection_params)
        
        elif source_system == 'SALESFORCE':
            return SourceConnector._extract_salesforce(query, connection_params)
        
        elif source_system == 'SHAREPOINT':
            return SourceConnector._extract_sharepoint(query, connection_params)
        
        elif source_system == 'SERVICENOW':
            return SourceConnector._extract_servicenow(query, connection_params)
        
        elif source_system == 'VOLUME':
            # Data already in Unity Catalog Volume - read directly
            return SourceConnector._extract_volume(query, connection_params)
        
        elif source_system == 'SFTP':
            return SourceConnector._extract_sftp(query, connection_params)
        
        else:
            raise ValueError(f"Unsupported source system: {source_system}")
    
    @staticmethod
    def _extract_jdbc(query, driver_type, params):
        """
        Generic JDBC extraction (Oracle, SAP, SQL Server, etc.)
        """
        # Connection details should come from Databricks Secrets or UC connections
        jdbc_url = params.get('jdbc_url') or dbutils.secrets.get(scope=f"{driver_type}_scope", key="jdbc_url")
        username = params.get('username') or dbutils.secrets.get(scope=f"{driver_type}_scope", key="username")
        password = params.get('password') or dbutils.secrets.get(scope=f"{driver_type}_scope", key="password")
        
        return (spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("query", query)
            .option("user", username)
            .option("password", password)
            .option("fetchsize", "10000")
            .option("numPartitions", "8")
            .load())
    
    @staticmethod
    def _extract_salesforce(query, params):
        """
        Salesforce extraction using SOQL or bulk API
        Requires databricks-salesforce connector or custom API integration
        """
        # Use Salesforce connector or REST API
        sf_username = dbutils.secrets.get(scope="salesforce_scope", key="username")
        sf_password = dbutils.secrets.get(scope="salesforce_scope", key="password")
        sf_token = dbutils.secrets.get(scope="salesforce_scope", key="security_token")
        
        return (spark.read
            .format("com.springml.spark.salesforce")
            .option("username", sf_username)
            .option("password", sf_password + sf_token)
            .option("soql", query)
            .load())
    
    @staticmethod
    def _extract_sharepoint(query, params):
        """
        SharePoint extraction - typically via Microsoft Graph API or mounted storage
        """
        # Implementation depends on SharePoint access pattern (Graph API, mounted drive, etc.)
        sharepoint_path = params.get('sharepoint_path')
        return spark.read.format("csv").option("header", "true").load(sharepoint_path)
    
    @staticmethod
    def _extract_servicenow(query, params):
        """
        ServiceNow extraction via REST API
        """
        # Use REST API with Spark REST connector or requests library with parallelization
        snow_instance = dbutils.secrets.get(scope="servicenow_scope", key="instance_url")
        snow_username = dbutils.secrets.get(scope="servicenow_scope", key="username")
        snow_password = dbutils.secrets.get(scope="servicenow_scope", key="password")
        
        # Custom implementation using requests + Spark parallelize for large datasets
        raise NotImplementedError("ServiceNow connector - implement REST API integration")
    
    @staticmethod
    def _extract_volume(query, params):
        """
        Extract from Unity Catalog Volume (files already in Databricks)
        """
        volume_path = params.get('volume_path')
        file_format = params.get('file_format', 'parquet')
        
        return spark.read.format(file_format).load(volume_path)
    
    @staticmethod
    def _extract_sftp(query, params):
        """
        SFTP extraction - mount or direct file transfer
        """
        sftp_path = params.get('sftp_path')
        # Implementation: Use paramiko or dbutils.fs.mount with SFTP
        raise NotImplementedError("SFTP connector - implement file transfer logic")

print("SourceConnector class loaded. Ready for multi-source extraction.")

In [0]:
# =====================================================
# Component 4: Audit Logger & Watermark Updater
# =====================================================
# Records execution metrics and updates watermark state

def start_extract_audit(object_id, job_run_id, load_incremental):
    """
    Create audit record at extraction start
    
    Returns:
        int: ExtractID for this execution
    """
    spark.sql(f"""
        INSERT INTO mde_dev.MetadataExplorer.ObjectsExtract (
            ObjectID, JobRunID, LoadIncremental, StartTime, ExtractionStatus
        )
        VALUES (
            {object_id}, 
            '{job_run_id}', 
            {load_incremental}, 
            CURRENT_TIMESTAMP(), 
            'IN_PROGRESS'
        )
    """)
    
    # Get the ExtractID that was just created
    result = spark.sql("""
        SELECT MAX(ExtractID) as ExtractID 
        FROM mde_dev.MetadataExplorer.ObjectsExtract
    """).collect()[0]
    
    return result.ExtractID

def complete_extract_audit(extract_id, rows_extracted, rows_copied, file_path, file_name, status='SUCCESS', error_msg=None):
    """
    Update audit record with completion metrics
    """
    error_clause = f"'{error_msg}'" if error_msg else "NULL"
    
    spark.sql(f"""
        UPDATE mde_dev.MetadataExplorer.ObjectsExtract
        SET 
            EndTime = CURRENT_TIMESTAMP(),
            Duration = CAST((UNIX_TIMESTAMP(CURRENT_TIMESTAMP()) - UNIX_TIMESTAMP(StartTime)) AS DECIMAL(10,2)),
            RowsExtracted = {rows_extracted},
            RowsCopied = {rows_copied},
            FilePath = '{file_path}',
            FileName = '{file_name}',
            ExtractionStatus = '{status}',
            ErrorMessage = {error_clause}
        WHERE ExtractID = {extract_id}
    """)

def update_watermark(object_id, new_watermark_value):
    """
    Update LastWatermarkValue after successful incremental load
    
    Args:
        object_id (int): ObjectID from ObjectsList
        new_watermark_value (str): New watermark timestamp or composite key
    """
    spark.sql(f"""
        UPDATE mde_dev.MetadataExplorer.ObjectsList
        SET 
            LastWatermarkValue = '{new_watermark_value}',
            LastModifiedDate = CURRENT_TIMESTAMP(),
            LastModifiedBy = CURRENT_USER()
        WHERE ObjectID = {object_id}
    """)
    
    print(f"✓ Watermark updated for ObjectID {object_id}: {new_watermark_value}")

def calculate_watermark(df, watermark_columns):
    """
    Calculate new watermark value from extracted DataFrame
    
    Args:
        df: Extracted DataFrame
        watermark_columns: List of watermark column names
    
    Returns:
        str: New watermark value (max value from dataset)
    """
    if not watermark_columns or df.count() == 0:
        return None
    
    if len(watermark_columns) == 1:
        # Single column watermark
        max_val = df.agg({watermark_columns[0]: "max"}).collect()[0][0]
        return str(max_val) if max_val else None
    else:
        # Composite watermark - concatenate with delimiter
        watermark_expr = concat_ws("_", *[col(c) for c in watermark_columns])
        max_val = df.select(watermark_expr.alias("wm")).agg({"wm": "max"}).collect()[0][0]
        return str(max_val) if max_val else None

print("Audit logging functions loaded.")

# Metadata-Driven Ingestion Framework - Control Plane

**Enterprise Lakehouse Architecture** | Built for heterogeneous source integration (Oracle, SAP, Salesforce, SharePoint, SFTP, OneDrive, ServiceNow, Volumes)

This notebook establishes the metadata control plane using Unity Catalog and Delta Lake for a dynamic, configuration-driven ingestion framework.

In [0]:
%sql
-- =====================================================
-- STEP 1: Unity Catalog & Schema Setup
-- =====================================================

-- Create the catalog for the Metadata-Driven Ingestion Framework
CREATE CATALOG IF NOT EXISTS mde_dev
COMMENT 'Metadata-Driven Ingestion Framework - Development Environment';

-- Set catalog for session context
USE CATALOG mde_dev;

-- Create the MetadataExplorer schema with governance and isolation
CREATE SCHEMA IF NOT EXISTS MetadataExplorer
COMMENT 'Control plane schema containing metadata tables for dynamic ingestion orchestration';

USE SCHEMA MetadataExplorer;

-- Verify creation
SELECT current_catalog(), current_schema();

In [0]:
%sql
-- =====================================================
-- STEP 2.1: ObjectsList - Source Object Registry
-- =====================================================
-- This table defines WHAT to extract, WHERE from, and HOW
-- Serves as the master configuration for all ingestion objects

CREATE TABLE IF NOT EXISTS mde_dev.MetadataExplorer.ObjectsList (
  ObjectID BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  ObjectSchema STRING COMMENT 'Source schema/namespace (e.g., DBO, PUBLIC, AR)',
  ObjectName STRING NOT NULL COMMENT 'Fully qualified source object name (e.g., SALESFORCE.Account, ORACLE.AR_CUSTOMERS)',
  SourceSystem STRING NOT NULL COMMENT 'Source system type: Oracle, SAP, Salesforce, SharePoint, SFTP, OneDrive, ServiceNow, Volume',
  IncludeInLoad INT NOT NULL DEFAULT 1 COMMENT 'Inclusion flag: 1=Active, 0=Excluded',
  LoadIncremental INT NOT NULL DEFAULT 0 COMMENT 'Load strategy: 0=Full, 1=Incremental',
  LoadGroup INT NOT NULL DEFAULT 1 COMMENT 'Parallel execution batch grouping ID',
  LoadOrder INT NOT NULL DEFAULT 1 COMMENT 'Sequential execution order within LoadGroup',
  TargetSchema STRING NOT NULL COMMENT 'Destination Unity Catalog schema (e.g., bronze_raw, bronze_erp)',
  TargetTableName STRING NOT NULL COMMENT 'Destination Delta table name',
  PrimaryKeys ARRAY<STRING> COMMENT 'Primary key column(s) for deduplication and merge logic',
  WatermarkColumn ARRAY<STRING> COMMENT 'Incremental tracking column(s) for CDC (e.g., LAST_UPDATE_DATE, MODIFIED_TIMESTAMP)',
  LastWatermarkValue STRING COMMENT 'Serialized state of last successful extraction watermark (ISO timestamp or composite key)',
  HardDelete INT DEFAULT 0 COMMENT 'Physical deletion flag: 1=Apply DELETE operations, 0=Ignore',
  SoftDelete INT DEFAULT 1 COMMENT 'Logical deletion flag: 1=Mark as deleted with tombstone, 0=No soft delete tracking',
  PhotonEnable INT DEFAULT 1 COMMENT 'Photon acceleration: 1=Enabled, 0=Disabled',
  StoragePath STRING COMMENT 'Target storage location: s3://, abfss://, or /Volumes/catalog/schema/volume/path',
  CreatedDate TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT 'Record creation timestamp',
  CreatedBy STRING DEFAULT CURRENT_USER() COMMENT 'User who created this configuration',
  LastModifiedDate TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT 'Last modification timestamp',
  LastModifiedBy STRING DEFAULT CURRENT_USER() COMMENT 'User who last modified this configuration',
  
  CONSTRAINT pk_ObjectsList PRIMARY KEY (ObjectID),
  CONSTRAINT chk_IncludeInLoad CHECK (IncludeInLoad IN (0, 1)),
  CONSTRAINT chk_LoadIncremental CHECK (LoadIncremental IN (0, 1)),
  CONSTRAINT chk_HardDelete CHECK (HardDelete IN (0, 1)),
  CONSTRAINT chk_SoftDelete CHECK (SoftDelete IN (0, 1)),
  CONSTRAINT chk_PhotonEnable CHECK (PhotonEnable IN (0, 1))
)
USING DELTA
COMMENT 'Master registry of source objects and their extraction configuration'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5',
  'quality' = 'gold',
  'domain' = 'metadata_control_plane'
);

In [0]:
%sql
-- =====================================================
-- STEP 2.2: ObjectFields - Column-Level Metadata
-- =====================================================
-- Defines column projection, data type casting, and key constraints
-- Enables dynamic SELECT statement generation with type safety

CREATE TABLE IF NOT EXISTS mde_dev.MetadataExplorer.ObjectFields (
  FieldID BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  ObjectID BIGINT NOT NULL COMMENT 'Foreign key reference to ObjectsList.ObjectID',
  ColumnName STRING NOT NULL COMMENT 'Source system column name (as-is from source schema)',
  TargetColumnName STRING NOT NULL COMMENT 'Target column name after standardization/normalization',
  DataType STRING NOT NULL COMMENT 'Target Spark SQL data type (STRING, INTEGER, DECIMAL(18,2), TIMESTAMP, DATE, BOOLEAN, etc.)',
  IncludeInLoad INT NOT NULL DEFAULT 1 COMMENT 'Column inclusion flag: 1=Include in extraction, 0=Exclude',
  PrimaryKey INT NOT NULL DEFAULT 0 COMMENT 'Primary key indicator: 1=Part of PK, 0=Not a PK column',
  OrdinalPosition INT NOT NULL COMMENT 'Source column ordinal position for maintaining schema order',
  
  CONSTRAINT pk_ObjectFields PRIMARY KEY (FieldID),
  CONSTRAINT fk_ObjectFields_ObjectsList FOREIGN KEY (ObjectID) REFERENCES mde_dev.MetadataExplorer.ObjectsList(ObjectID),
  CONSTRAINT chk_IncludeInLoad_Field CHECK (IncludeInLoad IN (0, 1)),
  CONSTRAINT chk_PrimaryKey CHECK (PrimaryKey IN (0, 1))
)
USING DELTA
COMMENT 'Column-level metadata for dynamic schema projection and type casting'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5',
  'quality' = 'gold',
  'domain' = 'metadata_control_plane'
);

In [0]:
%sql
-- =====================================================
-- STEP 2.3: ObjectsExtract - Audit & Lineage Tracking
-- =====================================================
-- Captures runtime metrics, file generation, and execution lineage
-- Enables observability, troubleshooting, and SLA monitoring

CREATE TABLE IF NOT EXISTS mde_dev.MetadataExplorer.ObjectsExtract (
  ExtractID BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  ObjectID BIGINT NOT NULL COMMENT 'Reference to ObjectsList.ObjectID',
  JobRunID STRING COMMENT 'Databricks Job Run ID or Delta Live Tables Update ID',
  LoadIncremental INT NOT NULL COMMENT 'Load type executed: 0=Full, 1=Incremental',
  StartTime TIMESTAMP NOT NULL COMMENT 'Extraction start timestamp',
  EndTime TIMESTAMP COMMENT 'Extraction completion timestamp',
  Duration DECIMAL(10,2) COMMENT 'Execution duration in seconds',
  RowsExtracted BIGINT COMMENT 'Total rows read from source system',
  RowsCopied BIGINT COMMENT 'Total rows successfully written to target',
  FilePath STRING COMMENT 'Storage directory path where data landed (S3/ADLS/Volume)',
  FileName STRING COMMENT 'Generated file name or Delta table version',
  ExtractionStatus STRING NOT NULL DEFAULT 'IN_PROGRESS' COMMENT 'Status: SUCCESS, FAILED, IN_PROGRESS, SKIPPED',
  ErrorMessage STRING COMMENT 'Detailed error stack trace if extraction failed',
  
  CONSTRAINT pk_ObjectsExtract PRIMARY KEY (ExtractID),
  CONSTRAINT fk_ObjectsExtract_ObjectsList FOREIGN KEY (ObjectID) REFERENCES mde_dev.MetadataExplorer.ObjectsList(ObjectID),
  CONSTRAINT chk_LoadIncremental_Extract CHECK (LoadIncremental IN (0, 1)),
  CONSTRAINT chk_ExtractionStatus CHECK (ExtractionStatus IN ('SUCCESS', 'FAILED', 'IN_PROGRESS', 'SKIPPED'))
)
USING DELTA
COMMENT 'Audit trail and lineage tracking for all extraction executions'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5',
  'quality' = 'gold',
  'domain' = 'metadata_control_plane'
);

In [0]:
%sql
-- =====================================================
-- STEP 3: Sample Metadata Population
-- =====================================================
-- Example configurations for common enterprise sources

-- Insert sample objects
INSERT INTO mde_dev.MetadataExplorer.ObjectsList (
  ObjectSchema, ObjectName, SourceSystem, IncludeInLoad, LoadIncremental, 
  LoadGroup, LoadOrder, TargetSchema, TargetTableName, PrimaryKeys, 
  WatermarkColumn, StoragePath
)
VALUES
  -- Oracle ERP Customer Master
  ('AR', 'AR_CUSTOMERS', 'Oracle', 1, 1, 1, 1, 'bronze_erp', 'oracle_ar_customers', 
   array('CUSTOMER_ID'), array('LAST_UPDATE_DATE'), 's3://lakehouse-bronze/erp/ar_customers/'),
   
  -- Salesforce Account (Full Load)
  ('SALESFORCE', 'Account', 'Salesforce', 1, 0, 1, 2, 'bronze_crm', 'sfdc_account', 
   array('Id'), NULL, 's3://lakehouse-bronze/crm/accounts/'),
   
  -- SAP Material Master (Incremental)
  ('MARA', 'MATERIAL', 'SAP', 1, 1, 2, 1, 'bronze_sap', 'sap_material_master', 
   array('MATNR', 'WERKS'), array('LAEDA'), '/Volumes/mde_dev/bronze_sap/material/'),
   
  -- ServiceNow Incidents
  ('INCIDENT', 'incident', 'ServiceNow', 1, 1, 2, 2, 'bronze_itsm', 'snow_incidents', 
   array('sys_id'), array('sys_updated_on'), 's3://lakehouse-bronze/itsm/incidents/'),
   
  -- SharePoint Documents (Volume-based)
  ('SHAREPOINT', 'ProjectDocs', 'SharePoint', 1, 0, 3, 1, 'bronze_unstructured', 'sharepoint_documents', 
   array('DocumentID'), NULL, '/Volumes/mde_dev/bronze_unstructured/sharepoint/');

-- Insert sample column definitions for Oracle AR_CUSTOMERS
INSERT INTO mde_dev.MetadataExplorer.ObjectFields (
  ObjectID, ColumnName, TargetColumnName, DataType, IncludeInLoad, PrimaryKey, OrdinalPosition
)
VALUES
  (1, 'CUSTOMER_ID', 'customer_id', 'BIGINT', 1, 1, 1),
  (1, 'CUSTOMER_NAME', 'customer_name', 'STRING', 1, 0, 2),
  (1, 'CUSTOMER_TYPE', 'customer_type', 'STRING', 1, 0, 3),
  (1, 'ACCOUNT_NUMBER', 'account_number', 'STRING', 1, 0, 4),
  (1, 'CREDIT_LIMIT', 'credit_limit', 'DECIMAL(18,2)', 1, 0, 5),
  (1, 'LAST_UPDATE_DATE', 'last_update_date', 'TIMESTAMP', 1, 0, 6),
  (1, 'CREATION_DATE', 'creation_date', 'TIMESTAMP', 1, 0, 7),
  (1, 'STATUS', 'status', 'STRING', 1, 0, 8);

-- Verify metadata setup
SELECT 
  o.ObjectID,
  o.ObjectName,
  o.SourceSystem,
  o.TargetSchema,
  o.TargetTableName,
  o.LoadIncremental,
  COUNT(f.FieldID) as ColumnCount
FROM mde_dev.MetadataExplorer.ObjectsList o
LEFT JOIN mde_dev.MetadataExplorer.ObjectFields f ON o.ObjectID = f.ObjectID
WHERE o.IncludeInLoad = 1
GROUP BY o.ObjectID, o.ObjectName, o.SourceSystem, o.TargetSchema, o.TargetTableName, o.LoadIncremental
ORDER BY o.LoadGroup, o.LoadOrder;